In [5]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [_here, *_here.parents]:
    if (candidate / "run_synthetic.py").exists() and (candidate / "fm_graphex").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate Flowette project root. "
        "Run this notebook from inside the Flowette project."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

In [2]:
import json
import importlib
import torch

import fm_graphex.evaluation.synthetic_metrics as synthetic_metrics
importlib.reload(synthetic_metrics)

import run_synthetic as synthetic_runner
importlib.reload(synthetic_runner)

parser = synthetic_runner.build_arg_parser()
args = parser.parse_args([])

In [3]:
args.task = 'ego'
args.seed = 2
args.data_root = "./spectre_data"
args.out = "./outputs/synthetic/ego"
args.checkpoint_dir = "./checkpoints/synthetic"

# Checkpoint behaviour
args.force_train_encoder = False
args.force_train_fm = False

# Encoder
args.enc_hidden = 64
args.enc_out = 32
args.enc_layers = 3
args.enc_lr = 1e-3

# Flow matching
args.fm_hidden_dim = 128
args.fm_batch_size = 32
args.fm_lr = 1e-3
args.lambda_x = 0.5
args.lambda_e = 0.5
args.beta_end = 1.0
args.grad_clip = 1.0

# Generation
args.gen_batch_size = 32
args.edge_budget_mode = "sample"
args.require_connected = True

# Evaluation
args.clustering_bins = 10
args.skip_eval = False
args.skip_viz = False

# Task-specific settings
args.enc_pretrain_epochs = 50
args.fm_layers = 3
args.fm_epochs = 100
args.alpha_fgw = 0.5
args.num_samples = 40
args.n_steps = 200
args.use_rings = True
args.remove_ring = False
args.ring_count_path = 'cycle_count_ego.txt'

args = synthetic_runner.apply_task_defaults(args)

In [6]:
generated_graphs, metrics = synthetic_runner.run(args)